# GNN Arb Live Trading Bot — Kalshi BTC Hourly Markets

Runs the convergence + GNN filter strategy live on Kalshi.

**Prerequisites**: Run `kalshi_gnn_arb.ipynb` first to train the GNN and save weights to `output/gnn_arb_model.pt`.

| § | Section | Purpose |
|---|---------|--------|
| 1 | Config & Auth | Kalshi API client, credentials, trading params |
| 2 | GNN Model | Load trained weights from backtest |
| 3 | Live Data | Coinbase spot poller, Kalshi websocket listener, causal sigma |
| 4 | Signal Detection | Convergence scan + GNN filter on live quotes |
| 5 | Execution | Risk preflight, smart limit orders, position tracking |
| 6 | Main Loop | Start/stop orchestrator |
| 7 | Controls | Kill switch, diagnostics, enable_live |

In [ ]:
# § 1 — Config, imports, Kalshi API client
from __future__ import annotations
import base64, hashlib, json, math, os, sqlite3, threading, time, warnings
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Optional, List, Dict, Any

import numpy as np
import pandas as pd
import requests
import torch
from torch import nn
import websocket

warnings.filterwarnings('ignore')

# ── Trading config ────────────────────────────────────────────────
CFG = {
    'mode': 'paper',               # 'paper' or 'live'
    'live_enabled': False,
    # model
    'model_path': 'output/gnn_arb_model.pt',
    'gnn_threshold': 0.3,
    # convergence detection
    'conv_max_secs': 3600,
    'conv_min_fair': 0.70,
    'conv_min_edge': 0.001,
    'sigma_uncertainty_discount': 0.10,
    'min_edge_cents': 0.1,
    # position sizing
    'max_position_per_strike': 5,
    'arb_max_dollars': 75.0,
    'max_concurrent_positions': 3,
    # fees
    'kalshi_fee_cap': 0.07,
    # polling intervals
    'spot_poll_sec': 2.0,
    'decision_interval_sec': 5.0,
    # websocket — dedicated WS endpoint per official docs
    'ws_url': 'wss://external-api-ws.kalshi.com/trade-api/ws/v2',
    'ws_reconnect_base_sec': 2.0,
    'ws_reconnect_max_sec': 60.0,
    # order execution
    'order_expiration_sec': 30,
    'order_buffer_cents': 2,       # aggressive limit above ask
    # risk
    'min_entry_price': 0.60,       # only buy deep ITM
    'max_entry_price': 0.97,
    'daily_loss_limit': -50.0,
    'max_total_exposure': 500.0,
    # data
    'db_path': 'output/gnn_arb_live.db',
    'event_series': ('KXBTC', 'KXBTCD'),
    'sigma_min_points': 15,
}

Path('output').mkdir(exist_ok=True)


def kalshi_fee(price: float) -> float:
    p = max(0.0, min(1.0, price))
    return min(CFG['kalshi_fee_cap'], 0.07 * p / 0.50)


# ── Kalshi API Client ─────────────────────────────────────────────
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding


def load_credentials(env_path='~/.kalshi/credentials.env'):
    creds = {}
    path = Path(env_path).expanduser()
    if not path.exists():
        print(f'Warning: no credentials file at {path}')
        return creds
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        creds[k.strip()] = v.strip()
    return creds


class KalshiClient:
    DEMO_URL = 'https://demo-api.kalshi.co/trade-api/v2'
    PROD_URL = 'https://api.elections.kalshi.com/trade-api/v2'

    def __init__(self, env='demo', key_id=None, private_key_path=None):
        assert env in ('demo', 'prod')
        self.env = env
        self.base_url = self.DEMO_URL if env == 'demo' else self.PROD_URL
        self._path_prefix = '/trade-api/v2'

        creds = load_credentials()
        if env == 'demo':
            self.key_id = key_id or creds.get('KALSHI_DEMO_KEY_ID')
            kp = private_key_path or creds.get('KALSHI_DEMO_PRIVATE_KEY_PATH')
        else:
            self.key_id = key_id or creds.get('KALSHI_PROD_KEY_ID')
            kp = private_key_path or creds.get('KALSHI_PROD_PRIVATE_KEY_PATH')

        self.private_key = None
        if kp:
            kp_path = Path(kp).expanduser()
            if kp_path.exists():
                with open(kp_path, 'rb') as f:
                    self.private_key = serialization.load_pem_private_key(
                        f.read(), password=None)

        self.session = requests.Session()

    def _sign(self, method, path):
        if not self.private_key or not self.key_id:
            return {}
        ts = str(int(time.time() * 1000))
        msg = (ts + method + path.split('?')[0]).encode('utf-8')
        sig = self.private_key.sign(
            msg,
            padding.PSS(mgf=padding.MGF1(hashes.SHA256()),
                        salt_length=padding.PSS.DIGEST_LENGTH),
            hashes.SHA256())
        return {
            'KALSHI-ACCESS-KEY': self.key_id,
            'KALSHI-ACCESS-SIGNATURE': base64.b64encode(sig).decode('utf-8'),
            'KALSHI-ACCESS-TIMESTAMP': ts,
        }

    def ws_auth_headers(self):
        headers = self._sign('GET', '/trade-api/ws/v2')
        headers['Content-Type'] = 'application/json'
        return headers

    def _get(self, path, params=None):
        url = self.base_url + path
        headers = self._sign('GET', self._path_prefix + path)
        r = self.session.get(url, headers=headers, params=params, timeout=10)
        r.raise_for_status()
        return r.json()

    def get_events(self, series_ticker=None, status='open', limit=100):
        params = {'status': status, 'limit': limit}
        if series_ticker:
            params['series_ticker'] = series_ticker
        return self._get('/events', params)

    def get_markets(self, event_ticker=None, series_ticker=None,
                    status='open', limit=100):
        params = {'status': status, 'limit': limit}
        if event_ticker:
            params['event_ticker'] = event_ticker
        if series_ticker:
            params['series_ticker'] = series_ticker
        return self._get('/markets', params)

    def get_market(self, ticker):
        return self._get(f'/markets/{ticker}')

    def get_orderbook(self, ticker, depth=10):
        return self._get(f'/markets/{ticker}/orderbook', {'depth': depth})

    def get_balance(self):
        return self._get('/portfolio/balance')

    def get_positions(self):
        return self._get('/portfolio/positions')


# ── Set up clients ────────────────────────────────────────────────
# Prod: unauthenticated, for REST fallback reads
kalshi_prod = KalshiClient(env='prod')
kalshi_prod.private_key = None
kalshi_prod.key_id = None

# Authenticated prod client — used for websocket auth + order placement
kalshi_live = None
try:
    _kl = KalshiClient(env='prod')
    if _kl.private_key and _kl.key_id:
        bal = _kl.get_balance()
        balance_cents = bal.get('balance', 0) if isinstance(bal, dict) else 0
        print(f'Live auth OK. Balance: ${balance_cents/100:.2f}')
        kalshi_live = _kl
    else:
        print('No prod credentials — live trading + websocket auth unavailable')
except Exception as e:
    print(f'Live auth failed: {e}')

# Verify prod market data works
try:
    test = kalshi_prod.get_events(series_ticker='KXBTCD', status='open', limit=3)
    print(f'Prod market data OK — {len(test.get("events", []))} events')
except Exception as e:
    print(f'Prod market data failed: {e}')

print(f'Mode: {CFG["mode"]}')
print(f'Websocket: {CFG["ws_url"]}')

In [9]:
# § 2 — GNN Model definition + load trained weights

class MessagePassingLayer(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim):
        super().__init__()
        self.msg = nn.Sequential(
            nn.Linear(node_dim + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim))
        self.update = nn.Sequential(
            nn.Linear(node_dim + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim))
        self.norm = nn.LayerNorm(node_dim)

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index[0], edge_index[1]
        m = self.msg(torch.cat([x[src], edge_attr], dim=-1))
        agg = torch.zeros(x.size(0), m.size(1), device=x.device, dtype=m.dtype)
        agg.index_add_(0, dst, m)
        cnt = torch.zeros(x.size(0), device=x.device, dtype=m.dtype)
        cnt.index_add_(0, dst, torch.ones(m.size(0), device=x.device, dtype=m.dtype))
        agg = agg / cnt.clamp_min(1.0).unsqueeze(-1)
        out = self.update(torch.cat([x, agg], dim=-1))
        return self.norm(out)


class KalshiArbGNN(nn.Module):
    def __init__(self, node_in=5, edge_in=6, hidden=64, n_layers=3):
        super().__init__()
        self.node_embed = nn.Linear(node_in, hidden)
        self.edge_embed = nn.Linear(edge_in, hidden)
        self.layers = nn.ModuleList(
            [MessagePassingLayer(hidden, hidden, hidden) for _ in range(n_layers)])
        head_in = 2 * hidden + hidden + edge_in
        self.edge_head = nn.Sequential(
            nn.Linear(head_in, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, 1))

    def forward(self, x, edge_index, edge_attr):
        h = self.node_embed(x)
        e = self.edge_embed(edge_attr)
        for layer in self.layers:
            h = layer(h, edge_index, e)
        src, dst = edge_index[0], edge_index[1]
        edge_repr = torch.cat([h[src], h[dst], e, edge_attr], dim=-1)
        return self.edge_head(edge_repr).squeeze(-1)


# Load trained model
gnn_model = None
model_path = Path(CFG['model_path'])
if model_path.exists():
    ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
    gnn_model = KalshiArbGNN(
        node_in=ckpt.get('node_in', 5),
        edge_in=ckpt.get('edge_in', 6),
        hidden=ckpt.get('hidden', 64),
        n_layers=ckpt.get('n_layers', 3))
    gnn_model.load_state_dict(ckpt['state_dict'])
    gnn_model.eval()
    print(f'GNN model loaded from {model_path}')
    if 'cfg_snapshot' in ckpt:
        print(f'  Training config: {ckpt["cfg_snapshot"]}')
else:
    print(f'No model at {model_path} — run kalshi_gnn_arb.ipynb first')
    print('Bot will run WITHOUT GNN filtering (classical convergence only)')

GNN model loaded from output/gnn_arb_model.pt
  Training config: {'adverse_selection_prob': 0.75, 'adverse_cost_cents': (2, 5), 'impact_multiplier': 2.0, 'latency_cost_cents': 3.0, 'min_edge_cents': 0.1, 'max_position_per_strike': 5, 'arb_max_dollars': 75.0, 'gnn_hidden': 64, 'gnn_layers': 3, 'gnn_epochs': 30, 'gnn_lr': 0.001, 'gnn_threshold': 0.3, 'conv_max_secs': 3600, 'conv_min_fair': 0.7, 'conv_min_edge': 0.001, 'sigma_uncertainty_discount': 0.1}


In [ ]:
# § 3 — Live Data Pipeline
#
# Three workers run in background threads:
#   1. Coinbase BTC spot price poller (every 2s)
#   2. Kalshi data feed — tries websocket first (ticker channel, push updates),
#      falls back to REST polling if websocket 403s (prod key may lack ws perms)
#   3. Event tracker — finds the nearest hourly KXBTC/D event

_LOCK = threading.Lock()

SPOT = {'price': None, 'ts': None, 'history': []}
BOOKS = {}   # ticker -> {yes_bid, yes_ask, floor, close_time, ts, ...}
TRACKED = {'event': None, 'close_time': None,
           'refreshed_at': None}
BOT_STATE = {'running': False, 'threads': [], 'log': [],
             'trades_this_session': 0, 'iter': 0}

_WS_STATE = {
    'connected': False,
    'subscribed_event': None,
    'ws': None,
    'reconnect_count': 0,
    'last_msg_ts': None,
    'msg_count': 0,
    'needs_resubscribe': False,
    'mode': 'websocket',  # 'websocket' or 'rest_fallback'
}


def _log(msg):
    ts = datetime.now(timezone.utc).strftime('%H:%M:%S')
    entry = f'[{ts}] {msg}'
    BOT_STATE['log'].append(entry)
    if len(BOT_STATE['log']) > 500:
        BOT_STATE['log'] = BOT_STATE['log'][-200:]


# ── Coinbase spot poller ──────────────────────────────────────────

def _coinbase_spot():
    r = requests.get('https://api.coinbase.com/v2/prices/BTC-USD/spot', timeout=5)
    r.raise_for_status()
    return float(r.json()['data']['amount'])


def _spot_poller():
    while BOT_STATE['running']:
        try:
            price = _coinbase_spot()
            now = datetime.now(timezone.utc)
            with _LOCK:
                SPOT['price'] = price
                SPOT['ts'] = now
                SPOT['history'].append((now, price))
                cutoff = now - timedelta(minutes=70)
                SPOT['history'] = [(t, p) for t, p in SPOT['history'] if t > cutoff]
        except Exception as e:
            _log(f'spot err: {e}')
        _sleep(CFG['spot_poll_sec'])


# ── Kalshi data feed (websocket with REST fallback) ───────────────

def _parse_ticker_msg(msg_data):
    """Parse a Kalshi ticker websocket message into BOOKS format."""
    tk = msg_data.get('market_ticker')
    if not tk:
        return

    def _dollars(key):
        v = msg_data.get(key)
        if v is None or v == '':
            return None
        try:
            return float(v)
        except (ValueError, TypeError):
            return None

    ya = _dollars('yes_ask_dollars')
    yb = _dollars('yes_bid_dollars')
    na = 1.0 - yb if yb is not None else None
    now = datetime.now(timezone.utc)

    with _LOCK:
        existing = BOOKS.get(tk, {})
        BOOKS[tk] = {
            'yes_bid': yb if yb is not None else existing.get('yes_bid'),
            'yes_ask': ya if ya is not None else existing.get('yes_ask'),
            'no_ask': na if na is not None else existing.get('no_ask'),
            'floor': existing.get('floor'),
            'close_time': existing.get('close_time'),
            'status': existing.get('status', 'active'),
            'ts': now,
            'volume': msg_data.get('volume_fp') or existing.get('volume'),
        }
        _WS_STATE['last_msg_ts'] = now
        _WS_STATE['msg_count'] += 1


def _seed_books_rest(event_ticker):
    """REST fetch to populate BOOKS with floor strikes + initial quotes."""
    try:
        mkts = kalshi_prod.get_markets(
            event_ticker=event_ticker, limit=200).get('markets', [])
        now = datetime.now(timezone.utc)
        with _LOCK:
            for m in mkts:
                tk = m.get('ticker')
                if not tk:
                    continue
                yb = m.get('yes_bid')
                ya = m.get('yes_ask')
                yb = float(yb) / 100 if yb is not None else None
                ya = float(ya) / 100 if ya is not None else None
                na = 1.0 - yb if yb is not None and yb > 0.01 else None
                BOOKS[tk] = {
                    'yes_bid': yb, 'yes_ask': ya, 'no_ask': na,
                    'floor': m.get('floor_strike'),
                    'close_time': m.get('close_time'),
                    'status': (m.get('status') or '').lower(),
                    'ts': now,
                    'volume': m.get('volume'),
                }
        _log(f'REST seed: {len(mkts)} markets for {event_ticker}')
    except Exception as e:
        _log(f'REST seed err: {e}')


def _ws_subscribe(ws, market_tickers):
    """Subscribe to ticker channel for specific market tickers."""
    sub_msg = {
        'id': 1,
        'cmd': 'subscribe',
        'params': {
            'channels': ['ticker'],
            'market_tickers': market_tickers,
        }
    }
    ws.send(json.dumps(sub_msg))
    _log(f'WS subscribed: {len(market_tickers)} tickers')


def _get_event_tickers(event_ticker):
    """Get all market tickers for an event via REST."""
    try:
        mkts = kalshi_prod.get_markets(
            event_ticker=event_ticker, limit=200).get('markets', [])
        return [m['ticker'] for m in mkts if m.get('ticker')]
    except Exception:
        return []


def _ws_listener():
    """Websocket thread. Connects, subscribes to ticker channel, processes
    push updates. Falls back to REST polling on persistent 403."""

    while BOT_STATE['running']:
        auth_client = kalshi_live
        if auth_client is None:
            _log('WS: no auth client — falling back to REST polling')
            _WS_STATE['mode'] = 'rest_fallback'
            _rest_poller()
            return

        headers = auth_client.ws_auth_headers()
        if not headers:
            _log('WS: no auth headers — falling back to REST polling')
            _WS_STATE['mode'] = 'rest_fallback'
            _rest_poller()
            return

        header_list = [f'{k}: {v}' for k, v in headers.items()]

        try:
            ws = websocket.create_connection(
                CFG['ws_url'], header=header_list, timeout=15)
            _WS_STATE['connected'] = True
            _WS_STATE['ws'] = ws
            _WS_STATE['reconnect_count'] = 0
            _WS_STATE['mode'] = 'websocket'
            _log('WS connected')

            # Subscribe to current event if tracked
            with _LOCK:
                event = TRACKED.get('event')
            if event:
                _seed_books_rest(event)
                tickers = _get_event_tickers(event)
                if tickers:
                    _ws_subscribe(ws, tickers)
                    _WS_STATE['subscribed_event'] = event

            # Message loop
            while BOT_STATE['running']:
                if _WS_STATE['needs_resubscribe']:
                    _WS_STATE['needs_resubscribe'] = False
                    with _LOCK:
                        new_event = TRACKED.get('event')
                    if new_event and new_event != _WS_STATE.get('subscribed_event'):
                        _seed_books_rest(new_event)
                        tickers = _get_event_tickers(new_event)
                        if tickers:
                            _ws_subscribe(ws, tickers)
                            _WS_STATE['subscribed_event'] = new_event

                try:
                    ws.settimeout(5.0)
                    raw = ws.recv()
                except websocket.WebSocketTimeoutException:
                    continue
                except websocket.WebSocketConnectionClosedException:
                    _log('WS closed by server')
                    break

                if not raw:
                    continue

                try:
                    data = json.loads(raw)
                except json.JSONDecodeError:
                    continue

                msg_type = data.get('type', '')
                if msg_type == 'ticker':
                    _parse_ticker_msg(data.get('msg', {}))
                elif msg_type == 'subscribed':
                    sid = data.get('msg', {}).get('sid')
                    _log(f'WS subscription confirmed, sid={sid}')
                elif msg_type == 'error':
                    _log(f'WS error: {data.get("msg", {})}')

        except Exception as e:
            err_str = str(e)
            _log(f'WS error: {err_str}')

            # 403 = API key lacks websocket permissions — fall back to REST
            if '403' in err_str:
                _log('WS 403 — prod key lacks websocket perms. '
                     'Falling back to REST polling. '
                     'Regenerate your prod API key on Kalshi to fix.')
                _WS_STATE['mode'] = 'rest_fallback'
                _WS_STATE['connected'] = False
                _WS_STATE['ws'] = None
                _rest_poller()
                return

        _WS_STATE['connected'] = False
        _WS_STATE['ws'] = None
        _WS_STATE['subscribed_event'] = None

        if not BOT_STATE['running']:
            break

        _WS_STATE['reconnect_count'] += 1
        delay = min(
            CFG['ws_reconnect_base_sec'] * (2 ** _WS_STATE['reconnect_count']),
            CFG['ws_reconnect_max_sec'])
        _log(f'WS reconnecting in {delay:.0f}s '
             f'(attempt {_WS_STATE["reconnect_count"]})')
        _sleep(delay)


def _rest_poller():
    """Fallback: poll Kalshi REST API for book quotes."""
    _log('REST poller active (6s interval)')
    while BOT_STATE['running']:
        with _LOCK:
            event = TRACKED.get('event')
        if not event:
            _sleep(2)
            continue
        try:
            mkts = kalshi_prod.get_markets(
                event_ticker=event, limit=200).get('markets', [])
            now = datetime.now(timezone.utc)
            with _LOCK:
                for m in mkts:
                    tk = m.get('ticker')
                    if not tk:
                        continue
                    yb = m.get('yes_bid')
                    ya = m.get('yes_ask')
                    yb = float(yb) / 100 if yb is not None else None
                    ya = float(ya) / 100 if ya is not None else None
                    na = 1.0 - yb if yb is not None and yb > 0.01 else None
                    BOOKS[tk] = {
                        'yes_bid': yb, 'yes_ask': ya, 'no_ask': na,
                        'floor': m.get('floor_strike'),
                        'close_time': m.get('close_time'),
                        'status': (m.get('status') or '').lower(),
                        'ts': now,
                        'volume': m.get('volume'),
                    }
                _WS_STATE['last_msg_ts'] = now
                _WS_STATE['msg_count'] += len(mkts)
        except Exception as e:
            _log(f'REST poll err: {e}')
        _sleep(6.0)


# ── Event tracker ─────────────────────────────────────────────────

def _event_tracker():
    """Find the nearest open hourly BTC event, update TRACKED."""
    while BOT_STATE['running']:
        try:
            now = datetime.now(timezone.utc)
            best_event = None
            best_close = None
            for series in CFG['event_series']:
                resp = kalshi_prod.get_events(
                    series_ticker=series, status='open', limit=50)
                for ev in resp.get('events', []):
                    et = ev.get('event_ticker', '')
                    mkts = ev.get('markets', [])
                    if not mkts:
                        continue
                    ct_str = mkts[0].get('close_time')
                    if not ct_str:
                        continue
                    from dateutil import parser as dtparser
                    ct = dtparser.isoparse(ct_str)
                    if ct.tzinfo is None:
                        ct = ct.replace(tzinfo=timezone.utc)
                    ttl = (ct - now).total_seconds()
                    if ttl < 120 or ttl > 7200:
                        continue
                    if best_close is None or ct < best_close:
                        best_event = et
                        best_close = ct

            with _LOCK:
                old_event = TRACKED.get('event')
                if best_event and best_event != old_event:
                    _log(f'Tracking: {best_event} closes {best_close}')
                    BOOKS.clear()
                    _WS_STATE['needs_resubscribe'] = True
                TRACKED['event'] = best_event
                TRACKED['close_time'] = best_close
                TRACKED['refreshed_at'] = now
        except Exception as e:
            _log(f'tracker err: {e}')
        _sleep(60)


# ── Helpers ───────────────────────────────────────────────────────

def _sleep(secs):
    end = time.time() + secs
    while time.time() < end and BOT_STATE['running']:
        time.sleep(0.2)


def _causal_sigma():
    with _LOCK:
        hist = list(SPOT['history'])
    if len(hist) < CFG['sigma_min_points']:
        return None
    prices = np.array([p for _, p in hist], dtype=float)
    lr = np.diff(np.log(prices))
    if len(lr) < 5:
        return None
    s = float(np.std(lr) * np.sqrt(525960))
    if not np.isfinite(s) or s <= 0:
        return None
    return s


print('Live data pipeline ready (websocket + REST fallback).')

In [11]:
# § 4 — Signal Detection: Convergence scan + GNN filter
#
# Builds a contract graph from live quotes, runs convergence scan,
# then optionally filters with the GNN.

CASH = 0


def _norm_cdf(x):
    return 0.5 * math.erfc(-x / math.sqrt(2))


@dataclass
class LiveSignal:
    ticker: str
    side: str            # 'yes' or 'no'
    strike: float
    price: float         # ask price
    fair_value: float
    edge: float          # fair - price - fee
    gnn_score: float     # 0-1, or -1 if no GNN
    secs_remaining: float


def _build_live_graph(spot, secs_remaining):
    """Build contract graph from current BOOKS for GNN scoring."""
    with _LOCK:
        books = {k: dict(v) for k, v in BOOKS.items()}

    rows = []
    for tk, b in books.items():
        if b.get('status') != 'active':
            continue
        if b.get('floor') is None or b.get('yes_ask') is None:
            continue
        ya = b['yes_ask']
        yb = b.get('yes_bid') or 0.0
        na = b.get('no_ask') or (1.0 - yb if yb > 0.01 else None)
        if ya <= 0.01 or ya >= 0.99 or na is None:
            continue
        rows.append({'ticker': tk, 'strike': float(b['floor']),
                     'yes_bid': yb, 'yes_ask': ya, 'no_ask': na})

    if len(rows) < 3:
        return None, None

    rows.sort(key=lambda r: r['strike'])
    n_strikes = len(rows)
    n_nodes = 1 + 2 * n_strikes

    src_list, dst_list, weights, attrs, meta = [], [], [], [], []

    for i, r in enumerate(rows):
        K, ya, yb = r['strike'], r['yes_ask'], r['yes_bid']
        na = r['no_ask']
        nb = max(0.01, 1.0 - ya)
        yes_node = 1 + i
        no_node = 1 + n_strikes + i
        dist = abs(spot - K) / max(100.0, spot * 0.001)
        time_frac = secs_remaining / 3600.0
        spread_yes = ya - yb if yb > 0 else ya
        spread_no = na - nb if nb > 0 else na

        def _add(s, d, rate, etype, price, fee, spread):
            if rate <= 0:
                return
            w = -math.log(rate)
            src_list.append(s); dst_list.append(d); weights.append(w)
            attrs.append([w, spread, fee, dist, time_frac, 50.0])
            meta.append({'type': etype, 'strike': K, 'price': price,
                         'strike_idx': i, 'ticker': r['ticker']})

        _add(CASH, yes_node, 1.0/ya, 'buy_yes', ya, kalshi_fee(ya), spread_yes)
        if yb > 0.01:
            _add(yes_node, CASH, yb, 'sell_yes', yb, kalshi_fee(yb), spread_yes)
        _add(CASH, no_node, 1.0/na, 'buy_no', na, kalshi_fee(na), spread_no)
        if nb > 0.01:
            _add(no_node, CASH, nb, 'sell_no', nb, kalshi_fee(nb), spread_no)

    if not src_list:
        return None, None

    node_attr = np.zeros((n_nodes, 5), dtype=np.float32)
    node_attr[0, 0] = 1.0
    strikes = [r['strike'] for r in rows]
    for i in range(n_strikes):
        node_attr[1 + i, 1] = 1.0
        node_attr[1 + i, 3] = (strikes[i] - spot) / max(100, spot * 0.001)
        node_attr[1 + i, 4] = abs(strikes[i] - spot) / max(100, spot * 0.001)
        node_attr[1 + n_strikes + i, 2] = 1.0
        node_attr[1 + n_strikes + i, 3] = (strikes[i] - spot) / max(100, spot * 0.001)
        node_attr[1 + n_strikes + i, 4] = abs(strikes[i] - spot) / max(100, spot * 0.001)

    graph_data = {
        'node_attr': node_attr,
        'edge_index': np.array([src_list, dst_list], dtype=np.int64),
        'edge_attr': np.array(attrs, dtype=np.float32),
        'edge_meta': meta,
    }
    return graph_data, rows


def scan_live_signals():
    """Scan current books for convergence opportunities, filter with GNN."""
    with _LOCK:
        spot = SPOT.get('price')
        close_time = TRACKED.get('close_time')

    if spot is None or close_time is None:
        return []

    now = datetime.now(timezone.utc)
    secs = (close_time - now).total_seconds()
    if secs < 10 or secs > CFG['conv_max_secs']:
        return []

    sigma = _causal_sigma()
    if sigma is None:
        return []

    sigma_c = sigma * (1.0 + CFG['sigma_uncertainty_discount'])
    sig_s = sigma_c / math.sqrt(365.25 * 24 * 3600)
    sig_rem = sig_s * spot * math.sqrt(secs)
    if sig_rem <= 0:
        return []

    # Build graph for GNN scoring
    graph_data, rows = _build_live_graph(spot, secs)
    gnn_scores = {}
    if graph_data is not None and gnn_model is not None:
        with torch.no_grad():
            scores = torch.sigmoid(gnn_model(
                torch.from_numpy(graph_data['node_attr']),
                torch.from_numpy(graph_data['edge_index']),
                torch.from_numpy(graph_data['edge_attr'])
            )).numpy()
        for e_idx, m in enumerate(graph_data['edge_meta']):
            key = (m['ticker'], m['type'])
            gnn_scores[key] = float(scores[e_idx])

    # Scan convergence
    signals = []
    with _LOCK:
        books = {k: dict(v) for k, v in BOOKS.items()}

    for tk, b in books.items():
        if b.get('status') != 'active' or b.get('floor') is None:
            continue
        K = float(b['floor'])
        ya = b.get('yes_ask')
        na = b.get('no_ask')
        d = abs(spot - K) / sig_rem
        fair_yes = _norm_cdf(d) if spot > K else 1 - _norm_cdf(d)

        # YES side
        if (fair_yes > CFG['conv_min_fair'] and ya is not None
                and 0.01 < ya < 0.99):
            fee = kalshi_fee(ya)
            edge = fair_yes - ya - fee
            if edge > CFG['conv_min_edge']:
                gs = gnn_scores.get((tk, 'buy_yes'), -1.0)
                signals.append(LiveSignal(
                    ticker=tk, side='yes', strike=K, price=ya,
                    fair_value=fair_yes, edge=edge, gnn_score=gs,
                    secs_remaining=secs))

        # NO side
        fair_no = 1.0 - fair_yes
        if (fair_no > CFG['conv_min_fair'] and na is not None
                and 0.01 < na < 0.99):
            fee = kalshi_fee(na)
            edge = fair_no - na - fee
            if edge > CFG['conv_min_edge']:
                gs = gnn_scores.get((tk, 'buy_no'), -1.0)
                signals.append(LiveSignal(
                    ticker=tk, side='no', strike=K, price=na,
                    fair_value=fair_no, edge=edge, gnn_score=gs,
                    secs_remaining=secs))

    # GNN filter
    if gnn_model is not None:
        signals = [s for s in signals
                   if s.gnn_score < 0 or s.gnn_score >= CFG['gnn_threshold']]

    signals.sort(key=lambda s: s.edge, reverse=True)
    return signals


print('Signal detection ready.')

Signal detection ready.


In [12]:
# § 5 — Execution: risk preflight, position tracking, order placement

# ── SQLite trade log ──────────────────────────────────────────────
def _db_conn():
    conn = sqlite3.connect(CFG['db_path'])
    conn.row_factory = sqlite3.Row
    return conn


def _init_db():
    conn = _db_conn()
    conn.executescript('''
    CREATE TABLE IF NOT EXISTS trades (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp_utc TEXT NOT NULL,
        event_ticker TEXT,
        market_ticker TEXT NOT NULL,
        side TEXT NOT NULL,
        contracts INTEGER NOT NULL,
        entry_price REAL NOT NULL,
        fair_value REAL,
        edge_cents REAL,
        gnn_score REAL,
        sigma REAL,
        spot REAL,
        secs_remaining REAL,
        mode TEXT DEFAULT 'paper',
        settled INTEGER DEFAULT 0,
        settlement_value REAL,
        pnl REAL,
        settled_at TEXT,
        order_id TEXT
    );
    CREATE INDEX IF NOT EXISTS idx_trades_settled
        ON trades(settled);
    CREATE INDEX IF NOT EXISTS idx_trades_ticker
        ON trades(market_ticker, settled);
    ''')
    conn.commit()
    conn.close()

_init_db()


# ── Position queries ──────────────────────────────────────────────
def _open_positions():
    conn = _db_conn()
    rows = conn.execute(
        'SELECT * FROM trades WHERE settled = 0').fetchall()
    conn.close()
    return [dict(r) for r in rows]


def _ticker_is_open(ticker):
    conn = _db_conn()
    n = conn.execute(
        'SELECT COUNT(*) FROM trades WHERE settled=0 AND market_ticker=?',
        (ticker,)).fetchone()[0]
    conn.close()
    return n > 0


def _open_exposure():
    conn = _db_conn()
    row = conn.execute(
        'SELECT COALESCE(SUM(entry_price * contracts), 0) '
        'FROM trades WHERE settled=0').fetchone()
    conn.close()
    return float(row[0])


def _today_pnl():
    today = datetime.now(timezone.utc).replace(
        hour=0, minute=0, second=0, microsecond=0).isoformat()
    conn = _db_conn()
    row = conn.execute(
        'SELECT COALESCE(SUM(pnl), 0) FROM trades '
        'WHERE settled=1 AND settled_at >= ?', (today,)).fetchone()
    conn.close()
    return float(row[0])


# ── Risk preflight ────────────────────────────────────────────────
def risk_preflight(signal: LiveSignal) -> tuple:
    """Returns (allowed: bool, reasons: list[str])."""
    reasons = []

    if _ticker_is_open(signal.ticker):
        reasons.append(f'already open on {signal.ticker}')
        return False, reasons

    if signal.price < CFG['min_entry_price']:
        reasons.append(f'price {signal.price:.2f} < min {CFG["min_entry_price"]}')
    if signal.price > CFG['max_entry_price']:
        reasons.append(f'price {signal.price:.2f} > max {CFG["max_entry_price"]}')

    n_open = len(_open_positions())
    if n_open >= CFG['max_concurrent_positions']:
        reasons.append(f'open {n_open} >= max {CFG["max_concurrent_positions"]}')

    exp = _open_exposure()
    cost = signal.price * CFG['max_position_per_strike']
    if exp + cost > CFG['max_total_exposure']:
        reasons.append(f'exposure ${exp+cost:.2f} > ${CFG["max_total_exposure"]}')

    pnl = _today_pnl()
    if pnl < CFG['daily_loss_limit']:
        reasons.append(f'daily PnL ${pnl:.2f} < limit ${CFG["daily_loss_limit"]}')

    return (len(reasons) == 0), reasons


# ── Order placement ───────────────────────────────────────────────
def _place_order(signal: LiveSignal) -> Optional[str]:
    """Place order. Returns order_id or None.

    Paper mode: records trade directly.
    Live mode: places aggressive limit order via kalshi_live.
    """
    contracts = min(
        CFG['max_position_per_strike'],
        max(1, int(CFG['arb_max_dollars'] / max(0.01, signal.price))))

    mode = CFG.get('mode', 'paper')
    order_id = None

    if mode == 'live':
        if kalshi_live is None:
            _log('SKIP: kalshi_live not initialized')
            return None
        try:
            buffer = CFG['order_buffer_cents'] / 100.0
            limit_price = min(0.99, signal.price + buffer)
            body = {
                'ticker': signal.ticker,
                'side': signal.side,
                'action': 'buy',
                'count': contracts,
                'type': 'limit',
                'client_order_id': f'gnn-{int(time.time()*1000)}',
                f'{signal.side}_price': int(round(limit_price * 100)),
                'expiration_ts': int(time.time()) + CFG['order_expiration_sec'],
            }
            h = kalshi_live._sign(
                'POST', kalshi_live._path_prefix + '/portfolio/orders')
            h['Content-Type'] = 'application/json'
            r = kalshi_live.session.post(
                kalshi_live.base_url + '/portfolio/orders',
                headers=h, json=body, timeout=10)
            if r.status_code >= 400:
                _log(f'ORDER FAILED {r.status_code}: {r.text[:200]}')
                return None
            resp = r.json()
            order_id = resp.get('order', {}).get('order_id')
            _log(f'LIVE ORDER: {signal.ticker} {signal.side} x{contracts} '
                 f'@ ${limit_price:.2f} id={order_id}')
        except Exception as e:
            _log(f'ORDER ERROR: {e}')
            return None

    # Record in DB
    conn = _db_conn()
    conn.execute(
        '''INSERT INTO trades(timestamp_utc, event_ticker, market_ticker,
           side, contracts, entry_price, fair_value, edge_cents,
           gnn_score, sigma, spot, secs_remaining, mode, order_id)
           VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?)''',
        (datetime.now(timezone.utc).isoformat(),
         TRACKED.get('event'), signal.ticker, signal.side,
         contracts, signal.price, signal.fair_value,
         signal.edge * 100, signal.gnn_score,
         _causal_sigma(), SPOT.get('price'),
         signal.secs_remaining, mode, order_id))
    conn.commit()
    conn.close()

    BOT_STATE['trades_this_session'] += 1
    _log(f'ENTRY: {signal.ticker} {signal.side} x{contracts} '
         f'@ ${signal.price:.2f}  fair={signal.fair_value:.3f}  '
         f'edge={signal.edge*100:.1f}c  gnn={signal.gnn_score:.2f}  '
         f'mode={mode}')
    return order_id or 'paper'


# ── Settlement checker ────────────────────────────────────────────
def check_settlements():
    """Check if any open positions have settled on Kalshi."""
    positions = _open_positions()
    if not positions:
        return

    for pos in positions:
        tk = pos['market_ticker']
        try:
            m = kalshi_prod.get_market(tk).get('market', {})
            status = (m.get('status') or '').lower()
            if status not in ('settled', 'finalized', 'closed'):
                continue

            result = m.get('result', '').lower()
            if pos['side'] == 'yes':
                payout = 1.0 if result == 'yes' else 0.0
            else:
                payout = 1.0 if result == 'no' else 0.0

            pnl = (payout - pos['entry_price']) * pos['contracts']
            fee = kalshi_fee(pos['entry_price']) * pos['contracts']
            pnl -= fee

            conn = _db_conn()
            conn.execute(
                'UPDATE trades SET settled=1, settlement_value=?, pnl=?, '
                'settled_at=? WHERE id=?',
                (payout, pnl, datetime.now(timezone.utc).isoformat(),
                 pos['id']))
            conn.commit()
            conn.close()
            _log(f'SETTLED: {tk} {pos["side"]} -> {result} '
                 f'PnL=${pnl:.2f}')
        except Exception as e:
            _log(f'settlement check err {tk}: {e}')


print('Execution engine ready.')

Execution engine ready.


In [13]:
# § 6 — Main Trading Loop

def _decision_loop():
    """Core loop: scan signals, check risk, execute."""
    while BOT_STATE['running']:
        BOT_STATE['iter'] += 1
        try:
            if BOT_STATE['iter'] % 12 == 0:
                check_settlements()

            signals = scan_live_signals()
            if not signals:
                _sleep(CFG['decision_interval_sec'])
                continue

            for sig in signals:
                allowed, reasons = risk_preflight(sig)
                if not allowed:
                    if BOT_STATE['iter'] % 60 == 0:
                        _log(f'SKIP {sig.ticker}: {reasons[0]}')
                    continue
                _place_order(sig)
                break  # one trade per cycle

        except Exception as e:
            _log(f'decision err: {e}')

        _sleep(CFG['decision_interval_sec'])


def start_bot():
    """Start spot poller, websocket listener, event tracker, decision loop."""
    if BOT_STATE['running']:
        print('Bot already running.')
        return

    BOT_STATE['running'] = True
    BOT_STATE['iter'] = 0
    BOT_STATE['log'] = []
    BOT_STATE['trades_this_session'] = 0
    _WS_STATE['msg_count'] = 0
    _WS_STATE['reconnect_count'] = 0

    workers = [
        ('spot_poller', _spot_poller),
        ('ws_listener', _ws_listener),
        ('event_tracker', _event_tracker),
        ('decision_loop', _decision_loop),
    ]
    threads = []
    for name, fn in workers:
        t = threading.Thread(target=fn, name=name, daemon=True)
        t.start()
        threads.append(t)
    BOT_STATE['threads'] = threads

    gnn_status = 'ENABLED' if gnn_model is not None else 'DISABLED'
    print(f'Bot started. Mode={CFG["mode"]}  GNN={gnn_status}  '
          f'Threads={len(threads)}')
    print(f'  Data: websocket (no polling)')
    print(f'  Max position: {CFG["max_position_per_strike"]} contracts, '
          f'${CFG["arb_max_dollars"]} cap')
    print(f'  Convergence: fair>{CFG["conv_min_fair"]:.0%}, '
          f'window={CFG["conv_max_secs"]}s')
    print(f'  GNN threshold: {CFG["gnn_threshold"]}')
    print(f'  Use status() to monitor, stop_bot() to halt.')


def stop_bot():
    """Stop all threads gracefully."""
    BOT_STATE['running'] = False
    # Close websocket if open
    ws = _WS_STATE.get('ws')
    if ws:
        try:
            ws.close()
        except Exception:
            pass
    for t in BOT_STATE.get('threads', []):
        t.join(timeout=5)
    BOT_STATE['threads'] = []
    _WS_STATE['connected'] = False
    _WS_STATE['ws'] = None
    print(f'Bot stopped. Trades this session: '
          f'{BOT_STATE["trades_this_session"]}')


def status():
    """Print current bot status."""
    print(f'Running: {BOT_STATE["running"]}  '
          f'Mode: {CFG["mode"]}  '
          f'Iter: {BOT_STATE["iter"]}')

    # Websocket status
    ws_connected = _WS_STATE.get('connected', False)
    ws_msgs = _WS_STATE.get('msg_count', 0)
    ws_event = _WS_STATE.get('subscribed_event')
    ws_last = _WS_STATE.get('last_msg_ts')
    ws_age = f'{(datetime.now(timezone.utc) - ws_last).total_seconds():.0f}s ago' if ws_last else 'never'
    print(f'WS: {"connected" if ws_connected else "disconnected"}  '
          f'msgs={ws_msgs}  sub={ws_event}  last={ws_age}')

    with _LOCK:
        spot = SPOT.get('price')
        spot_ts = SPOT.get('ts')
        event = TRACKED.get('event')
        close = TRACKED.get('close_time')
        n_books = len(BOOKS)

    print(f'Spot: ${spot:,.2f}' if spot else 'Spot: unavailable',
          end='')
    if spot_ts:
        age = (datetime.now(timezone.utc) - spot_ts).total_seconds()
        print(f' ({age:.0f}s ago)')
    else:
        print()

    if event:
        ttl = (close - datetime.now(timezone.utc)).total_seconds() / 60
        print(f'Event: {event}  TTL: {ttl:.1f}min  Books: {n_books}')
    else:
        print('Event: none tracked')

    sigma = _causal_sigma()
    print(f'Sigma: {sigma*100:.1f}%' if sigma else 'Sigma: insufficient data')

    positions = _open_positions()
    print(f'\nOpen positions: {len(positions)}')
    for p in positions:
        print(f'  {p["market_ticker"]} {p["side"]} x{p["contracts"]} '
              f'@ ${p["entry_price"]:.2f}  '
              f'edge={p["edge_cents"]:.1f}c  gnn={p["gnn_score"]:.2f}')

    print(f'\nToday PnL: ${_today_pnl():.2f}  '
          f'Exposure: ${_open_exposure():.2f}')
    print(f'Trades this session: {BOT_STATE["trades_this_session"]}')

    recent = BOT_STATE['log'][-5:]
    if recent:
        print(f'\nRecent log:')
        for entry in recent:
            print(f'  {entry}')


print('Main loop ready.')
print('  start_bot()   — start websocket + trading')
print('  stop_bot()    — stop all threads')
print('  status()      — current state')

Main loop ready.
  start_bot()   — start websocket + trading
  stop_bot()    — stop all threads
  status()      — current state


In [14]:
# § 7 — Kill Switch, Diagnostics, Live Mode

def kill_switch():
    """Emergency stop. Halts bot, cancels orders, switches to paper."""
    print('KILL SWITCH activated...')
    CFG['mode'] = 'paper'
    CFG['live_enabled'] = False
    stop_bot()

    if kalshi_live:
        try:
            orders = kalshi_live._get(
                '/portfolio/orders',
                {'status': 'resting'}).get('orders', []) or []
            for o in orders:
                oid = o.get('order_id')
                h = kalshi_live._sign(
                    'DELETE',
                    kalshi_live._path_prefix + f'/portfolio/orders/{oid}')
                kalshi_live.session.delete(
                    kalshi_live.base_url + f'/portfolio/orders/{oid}',
                    headers=h, timeout=10)
            print(f'  Canceled {len(orders)} resting orders')
        except Exception as e:
            print(f'  Order cancel error: {e}')

    kalshi_prod.session = requests.Session()
    if kalshi_live:
        kalshi_live.session = requests.Session()
    print('  All activity halted. Mode = paper.')


def enable_live():
    """Switch to live trading mode."""
    if kalshi_live is None:
        print('kalshi_live not initialized — set prod credentials first')
        return
    try:
        bal = kalshi_live.get_balance()
        balance = float(bal.get('balance', 0)) / 100.0
    except Exception as e:
        print(f'Cannot read balance: {e}')
        return
    if balance <= 0:
        print('Balance is $0 — refusing to enable')
        return
    CFG['mode'] = 'live'
    CFG['live_enabled'] = True
    print(f'LIVE TRADING ENABLED')
    print(f'  Balance: ${balance:.2f}')
    print(f'  Max per trade: ${CFG["arb_max_dollars"]:.2f}')
    print(f'  Max exposure: ${CFG["max_total_exposure"]:.2f}')
    print(f'  disable_live() to switch back to paper')


def disable_live():
    """Switch back to paper mode."""
    CFG['mode'] = 'paper'
    CFG['live_enabled'] = False
    print('Live trading disabled. Mode = paper.')


def diagnostics():
    """Detailed diagnostics for debugging."""
    print('=== DIAGNOSTICS ===')
    print(f'Mode: {CFG["mode"]}  Live: {CFG["live_enabled"]}')
    print(f'Running: {BOT_STATE["running"]}  Iter: {BOT_STATE["iter"]}')

    # Thread health
    alive = [t.name for t in BOT_STATE.get('threads', []) if t.is_alive()]
    dead = [t.name for t in BOT_STATE.get('threads', []) if not t.is_alive()]
    print(f'\nThreads alive: {alive}')
    if dead:
        print(f'Threads DEAD: {dead}')

    # Websocket health
    print(f'\nWebsocket:')
    print(f'  Connected: {_WS_STATE.get("connected", False)}')
    print(f'  Subscribed to: {_WS_STATE.get("subscribed_event")}')
    print(f'  Messages received: {_WS_STATE.get("msg_count", 0)}')
    print(f'  Reconnect count: {_WS_STATE.get("reconnect_count", 0)}')
    ws_last = _WS_STATE.get('last_msg_ts')
    if ws_last:
        age = (datetime.now(timezone.utc) - ws_last).total_seconds()
        print(f'  Last message: {age:.1f}s ago')
    else:
        print(f'  Last message: never')

    # Data freshness
    now = datetime.now(timezone.utc)
    with _LOCK:
        spot_age = (now - SPOT['ts']).total_seconds() if SPOT.get('ts') else None
        n_hist = len(SPOT.get('history', []))
        n_books = len(BOOKS)
        book_ages = [(tk, (now - b['ts']).total_seconds())
                     for tk, b in BOOKS.items() if b.get('ts')]

    print(f'\nSpot age: {spot_age:.1f}s' if spot_age else '\nSpot: no data')
    print(f'Spot history: {n_hist} points')
    print(f'Books: {n_books} tickers')
    if book_ages:
        max_age = max(a for _, a in book_ages)
        min_age = min(a for _, a in book_ages)
        print(f'  Book age range: {min_age:.1f}s - {max_age:.1f}s')

    sigma = _causal_sigma()
    print(f'Sigma: {sigma*100:.2f}%' if sigma else 'Sigma: None')

    # Signal scan
    signals = scan_live_signals()
    print(f'\nCurrent signals: {len(signals)}')
    for s in signals[:5]:
        print(f'  {s.ticker} {s.side} @ ${s.price:.2f}  '
              f'fair={s.fair_value:.3f}  edge={s.edge*100:.1f}c  '
              f'gnn={s.gnn_score:.2f}')

    # Trade history
    conn = _db_conn()
    total = conn.execute('SELECT COUNT(*) FROM trades').fetchone()[0]
    settled = conn.execute(
        'SELECT COUNT(*) FROM trades WHERE settled=1').fetchone()[0]
    total_pnl = conn.execute(
        'SELECT COALESCE(SUM(pnl),0) FROM trades WHERE settled=1'
    ).fetchone()[0]
    wins = conn.execute(
        'SELECT COUNT(*) FROM trades WHERE settled=1 AND pnl>0'
    ).fetchone()[0]
    conn.close()

    print(f'\nAll-time: {total} trades, {settled} settled')
    if settled > 0:
        print(f'  PnL: ${total_pnl:.2f}  Win rate: {wins/settled*100:.0f}%')

    # Full log
    print(f'\nRecent log ({len(BOT_STATE["log"])} entries):')
    for entry in BOT_STATE['log'][-15:]:
        print(f'  {entry}')


def trade_history(n=20):
    """Show recent trades."""
    conn = _db_conn()
    rows = conn.execute(
        'SELECT * FROM trades ORDER BY id DESC LIMIT ?', (n,)).fetchall()
    conn.close()
    if not rows:
        print('No trades yet.')
        return
    df = pd.DataFrame([dict(r) for r in rows])
    cols = ['id', 'timestamp_utc', 'market_ticker', 'side', 'contracts',
            'entry_price', 'edge_cents', 'gnn_score', 'mode', 'settled', 'pnl']
    cols = [c for c in cols if c in df.columns]
    print(df[cols].to_string(index=False))


print('Controls ready.')
print('  kill_switch()   — emergency stop')
print('  enable_live()   — switch to real money')
print('  disable_live()  — back to paper')
print('  diagnostics()   — detailed state dump')
print('  trade_history() — recent trades')

Controls ready.
  kill_switch()   — emergency stop
  enable_live()   — switch to real money
  disable_live()  — back to paper
  diagnostics()   — detailed state dump
  trade_history() — recent trades


## Run Book

### Paper trading (default)
1. Run all cells above (§1-§7)
2. `start_bot()` — begins polling spot + books, scanning for convergence signals
3. `status()` — check what it's doing
4. `diagnostics()` — detailed state dump
5. `stop_bot()` — stop when done

### Live trading
1. Run all cells above
2. `start_bot()` — start in paper mode first, verify signals look right
3. `enable_live()` — switch to real orders (requires prod credentials)
4. `status()` — monitor
5. `kill_switch()` — emergency stop, cancels all orders

In [15]:
# Start the bot (paper mode by default)
start_bot()

Bot started. Mode=paper  GNN=ENABLED  Threads=4
  Data: websocket (no polling)
  Max position: 5 contracts, $75.0 cap
  Convergence: fair>70%, window=3600s
  GNN threshold: 0.3
  Use status() to monitor, stop_bot() to halt.


In [16]:
status()

Running: True  Mode: paper  Iter: 2
WS: disconnected  msgs=0  sub=None  last=never
Spot: $79,848.74 (0s ago)
Event: none tracked
Sigma: insufficient data

Open positions: 0

Today PnL: $0.00  Exposure: $0.00
Trades this session: 0

Recent log:
  [00:48:24] WS error: Handshake status 403 Forbidden
  [00:48:24] WS reconnecting in 4s (attempt 1)
  [00:48:29] WS error: Handshake status 403 Forbidden
  [00:48:29] WS reconnecting in 8s (attempt 2)


In [ ]:
diagnostics()

In [ ]:
trade_history()

In [17]:
stop_bot()

Bot stopped. Trades this session: 0


In [ ]:
# Uncomment to enable live trading:
# enable_live()
# start_bot()

In [ ]:
# Emergency stop:
# kill_switch()